# 4.1. Matrix Factorization(MF) 기반 추천

추천을 위한 다양한 알고리즘 분류를 해보면, 크게 메모리 기반 알고리즘과 모델기반 알고리즘으로 나눌 수 있음.

|  | 메모리 기반 알고리즘 | 모델 기반 알고리즘 |
|-------|-------|-------|
| 설명 | 메모리에 있는 데이터를 계산해서 추천하는 방식 | 데이터로부터 미리 모델을 구성 후, 필요시 추천하는 방식 |
| 특징 | 개발 사용자 데이터 집중 | 전체 사용자 패턴 집중 |
| 장점 | 원래 데이터를 충실하게 사용 | 대규모 데이터에 빠르게 반응 |
| 단점 | 대규모 데이터에 느리게 반응 | 모델 생성 과정이 오래 걸림 |
| 예시 | CF | MF, Deep Learning |

MF 알고리즘은 아이템과 유저로 구성된 행렬을 두 개의 행렬로 분해하는 방식임. CF 방식과의 차이가 여기서 드러나는데, CF 방식에서는 아이템-유저 행렬을 full matrix로 활용했었음.

각 아이템과 유저 행렬은 K개의 잠재 요인(latent factor)으로 이루어져 있음.

![MF 알고리즘](../static/img_3.png)

예를 들어, K=2인 경우를 생각해보면 사용자, 아이템에 대한 latent matrix는 아래처럼 구상해볼 수 있음.
- 액션-멜로에 대한 잠재요인 (-1~1)
- 판타지-사실주의에 대한 잠재요인 (-1~1)

# 4.2. SGD(Stochastic Gradient Decent)를 사용한 MF 알고리즘

**MF 알고리즘 개념적 설명**

1. **잠재요인 K 설정**: 도메인에 따라 K가 어느정도가 좋을지 결정할 수도 있겠지만, 일반적으로 여러 개의 K 값을 할당해서 비교해가면서 실험하여, 최적의 K를 찾아냄.
2. **P,Q 행렬 초기화**
3. **예측 평점 R_hat 계산**: R_hat= P x Q^T
4. **실제 R과 R_hat간 오차 계산 및 P,Q 수정**: 오차를 줄이는 방향으로 P,Q를 수정하는 것이 MF의 핵심
5. **기준 오차 도달 확인**: 결과에 따라 다시 3번부터 수행

**SGD를 사용하는 이유**

P, Q를 한 번에 정확히 구하는 건 어려움 & 평점 데이터는 희소(sparse)

👉 관측된 평점만 하나씩 보면서 조금씩 고치는 것이 SGD 방식.

**SGD 기반 MF 학습 과정**

실제 평점과 예측 평점의 차이를 최소화

- 예측 평점
$$
\hat{r}_{ui} = p_u^{T} q_i
$$

- 오차
$$
e_{ui} = r_{ui} - \hat{r}_{ui}
$$

- 파라미터 업데이트 
$$
\begin{aligned}
p_u &\leftarrow p_u
+ \alpha ( e_{ui} q_i - \lambda p_u ) \\
q_i &\leftarrow q_i
+ \alpha ( e_{ui} p_u - \lambda q_i )
\end{aligned}
$$
$$
\alpha : \text{learning rate}, \quad
\lambda : \text{regularization coefficient (to prevent overfitting)}
$$


**Overfitting이 생기는 이유**

직관적으로
- 평점 데이터는 적은데
- 잠재 요인은 많음

👉 모델이 훈련 데이터만 외워버림

이 경우, 모델이 특정 사용자-아이템 조합은 잘 맞히지만, 새로운 데이터에는 엉망인 결과를 가져올 수 있음.

Overfitting을 방지(Regularization)하기 위해 등장하는 것이 추가 계수 $\lambda$

- 손실 함수: 원래 항에 추가적인 항을 붙인 형태가 됨
$$
\mathcal{L} =
\sum_{(u,i)\in\mathcal{D}}
\left(r_{ui} - p_u^{T} q_i\right)^2
+
\lambda
\left(
\|p_u\|^2 + \|q_i\|^2
\right)
$$

$\lambda$항은 파라미터를 작고 부드럽게 유지해주는 역할. 

그래서 SGD의 업데이트 과정을 보면, $\lambda$항이 작용하고 있음을 볼 수 있음.

$$
p_u \leftarrow p_u
+ \alpha
\left(
e_{ui} q_i - \lambda p_u
\right)
$$

$\lambda$가 너무 작으면 Overfitting으로 이어질 수 있고, 너무 크면 Underfitting으로 이어질 수 있기에 적절한 크기로 상정하는 것이 중요.

# 4.3. SGD를 활용한 기본 MF 알고리즘

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

base_src = Path.cwd().parent / 'data'
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')
# timestamp 제거
ratings = ratings[['user_id','movie_id','rating']].astype(int)

In [ ]:
class MF():
  def __init__(self,ratings,hyper_params):
    self.R = np.array(ratings)
    self.num_users,self.num_items = np.shape(self.R)
    self.K = hyper_params['K']
    self.alpha = hyper_params['alpha']
    self.beta = hyper_params['beta']
    self.iterations = hyper_params['iterations']
    self.verbose = hyper_params['verbose']

  def rmse(self):
    xs,ys = self.R.nonzero()
    self.predictions = []
    self.errors = []

    for x,y in zip(xs,ys):
      prediction = self.get_prediction(x,y)
      self.predictions.append(prediction)
      self.errors.append(self.R[x,y] - prediction)
    self.predictions = np.array(self.predictions)
    self.errors = np.array(self.errors)

    return np.sqrt(np.mean(self.errors**2))

  def train(self):
    self.P = np.random.normal(scale=1./self.K,
                              size=(self.num_users,self.K))
    self.Q = np.random.normal(scale=1./self.K,
                              size=(self.num_items,self.K))

    self.b_u = np.zeros(self.num_users)
    self.b_d = np.zeros(self.num_items)
    self.b = np.mean(self.R[self.R.nonzero()])

    rows,columns = self.R.nonzero()
    self.samples = [(i,j,self.R[i,j]) for i,j in zip(rows,columns)]

    training_process = []
    for i in range(self.iterations):
      np.random.shuffle(self.samples)
      self.sgd()
      rmse = self.rmse()
      training_process.append((i+1,rmse))
      if self.verbose:
        if (i+1) % 10 ==0:
          print('Iteration : %d ; train RMSE = %.4f'%(i+1,rmse))
    return training_process

  def get_prediction(self,i,j):
    # 평점 예측 = 평균 평점 + 사용자 편향 + 아이템 편향 + 잠재 요인 벡터의 내적
    prediction = self.b + self.b_u[i] + self.b_d[j] + self.P[i,:].dot(self.Q[j,].T)
    return prediction

  def sgd(self):
    for i,j,r in self.samples:
      prediction = self.get_prediction(i,j)
      e = (r-prediction)

      self.b_u[i] += self.alpha * (e - (self.beta * self.b_u[i]))
      self.b_d[j] += self.alpha * (e - (self.beta * self.b_d[j]))

      self.P[i,:] += self.alpha * ((e * self.Q[j,:]) - (self.beta * self.P[i,:]))
      self.Q[j,:] += self.alpha * ((e * self.P[i,:]) - (self.beta * self.Q[j,:]))

R_temp = ratings.pivot(index='user_id',
                       columns='movie_id',
                       values='rating').fillna(0)

hyper_params = {
    'K' : 30,
    'alpha' : 0.001,
    'beta' : 0.02,
    'iterations' : 100,
    'verbose' : True
}

mf = MF(R_temp, hyper_params)

train_process = mf.train()

Iteration : 10 ; train RMSE = 0.9585
Iteration : 20 ; train RMSE = 0.9373
Iteration : 30 ; train RMSE = 0.9280
Iteration : 40 ; train RMSE = 0.9225
Iteration : 50 ; train RMSE = 0.9183
Iteration : 60 ; train RMSE = 0.9144
Iteration : 70 ; train RMSE = 0.9098
Iteration : 80 ; train RMSE = 0.9035
Iteration : 90 ; train RMSE = 0.8947
Iteration : 100 ; train RMSE = 0.8828


# 4.4 train/test 분리 MF 알고리즘

In [4]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

base_src = Path.cwd().parent / 'data'
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')
# timestamp 제거
ratings = ratings[['user_id','movie_id','rating']].astype(int)


# train / test set 분리
from sklearn.utils import shuffle
TRAIN_SIZE = 0.75
# (사용자 - 영화 - 평점)
ratings = shuffle(ratings,random_state=2021)
cutoff = int(TRAIN_SIZE * len(ratings))
ratings_train = ratings.iloc[:cutoff]
ratings_test = ratings.iloc[cutoff:]


class NEW_MF():
  def __init__(self,ratings,hyper_params):
    self.R = np.array(ratings)
    # 사용자 수(num_users)와 아이템 수(num_iterms)를 받아온다.
    self.num_users,self.num_items = np.shape(self.R)
    # 아래는 MF weight 조절을 위한 하이퍼파라미터이다.
    # K : 잠재요인(latent factor)의 수
    self.K = hyper_params['K']
    # alpha : 학습률
    self.alpha = hyper_params['alpha']
    # beta : 정규화 계수
    self.beta = hyper_params['beta']
    # iterations : SGD의 계산을 할 때의 반복 횟수
    self.iterations = hyper_params['iterations']
    # verbose : SGD의 학습 과정을 중간중간에 출력할 것인지에 대한 여부
    self.verbose = hyper_params['verbose']

    item_id_index = []
    index_item_id = []
    for i, one_id in enumerate(ratings):
      item_id_index.append([one_id,i])
      index_item_id.append([i,one_id])
    self.item_id_index = dict(item_id_index)
    self.index_item_id = dict(index_item_id)

    user_id_index = []
    index_user_id = []
    for i,one_id in enumerate(ratings.T):
      user_id_index.append([one_id,i])
      index_user_id.append([i,one_id])
    self.user_id_index = dict(user_id_index)
    self.index_user_id = dict(index_user_id)


  def rmse(self):
    # self.R에서 평점이 있는(0이 아닌) 요소의 인덱스를 가져온다.
    xs, ys = self.R.nonzero()
    # prediction과 error를 담을 리스트 변수 초기화
    self.predictions = []
    self.errors = []
    # 평점이 있는 요소(사용자 x, 아이템 y) 각각에 대해서 아래의 코드를 실행한다.
    for x,y in zip(xs,ys):
      # 사용자 x, 아이템 y에 대해서 평점 예측치를 get_prediction() 함수를 사용해서 계산한다.
      prediction = self.get_prediction(x,y)
      # 예측값을 예측값 리스트에 추가한다.
      self.predictions.append(prediction)
      # 실제값(R)과 예측값의 차이(errors) 계산해서 오차값 리스트에 추가한다.
      self.errors.append(self.R[x,y] - prediction)
    # 예측값 리스트와 오차값 리스트를 numpy array형태로 변환한다.
    self.predictions = np.array(self.predictions)
    self.errors = np.array(self.errors)
    # error를 활용해서 RMSE 도출
    return np.sqrt(np.mean(self.errors**2))


  def sgd(self):
    for i,j,r in self.samples:
      # 사용자 i, 아이템 j에 대한 평점 예측치 계산
      prediction = self.get_prediction(i,j)
      # 실제 평점과 비교한 오차 계산
      e = (r - prediction)

      # 사용자 평가 경향 계산 및 업데이트
      self.b_u[i] += self.alpha * (e - (self.beta * self.b_u[i]))
      # 아이템 평가 경향 계산 및 업데이트
      self.b_d[j] += self.alpha * (e - (self.beta * self.b_d[j]))

      # P 행렬 계산 및 업데이트
      self.P[i,:] += self.alpha * ((e * self.Q[j,:]) - (self.beta * self.P[i,:]))
      # Q 행렬 계산 및 업데이트
      self.Q[j,:] += self.alpha * ((e * self.P[i,:]) - (self.beta * self.Q[j,:]))

  def get_prediction(self,i,j):
    # 사용자 i, 아이템 j에 대한 평점 예측치를 앞에서 배웠던 식을 이용해서 구한다.
    prediction = self.b + self.b_u[i] + self.b_d[j] + self.P[i,:].dot(self.Q[j,:].T)
    return prediction


  # Test set 선정
  def set_test(self,ratings_test):
    test_set = []
    for i in range(len(ratings_test)):
      x = self.user_id_index[ratings_test.iloc[i,0]]
      y = self.item_id_index[ratings_test.iloc[i,1]]
      z = ratings_test.iloc[i,2]
      test_set.append([x,y,z])
      self.R[x,y] = 0
    self.test_set = test_set
    return test_set


  # Test set RMSE 계산
  def test_rmse(self):
    error = 0
    for one_set in self.test_set:
      predicted = self.get_prediction(one_set[0],one_set[1])
      # e => e^2
      error += pow(one_set[2] - predicted,2)
    return np.sqrt(error/len(self.test_set))

  def test(self):
    self.P = np.random.normal(scale=1./self.K,
                              size=(self.num_users,self.K))
    self.Q = np.random.normal(scale=1./self.K,
                              size=(self.num_items,self.K))

    self.b_u = np.zeros(self.num_users)
    self.b_d = np.zeros(self.num_items)
    self.b = np.mean(self.R[self.R.nonzero()])

    rows,columns = self.R.nonzero()
    self.samples = [(i,j,self.R[i,j]) for i,j in zip(rows,columns)]

    training_process = []
    for i in range(self.iterations):
      np.random.shuffle(self.samples)
      self.sgd()
      rmse1 = self.rmse()
      rmse2 = self.test_rmse()
      training_process.append((i+1,rmse1,rmse2))
      if self.verbose:
        if (i+1) % 10 == 0:
          print('Iteration : %d ; Train RMSE = %.4f ; Test RMSE = %.4f'% (i+1 ,rmse1,rmse2))
    return training_process

  def get_one_prediction(self,user_id,item_id):
    return self.get_prediction(self.user_id_index[user_id],
                               self.item_id_index[item_id])
  def full_prediction(self):
    return self.b + self.b_u[:,np.newaxis] + self.b_d[np.newaxis,:] + self.P.dot(self.Q.T)

R_temp = ratings.pivot(index='user_id',
                       columns='movie_id',
                       values='rating').fillna(0)

hyper_params = {
    'K':30,
    'alpha':0.001,
    'beta':0.02,
    'iterations':100,
    'verbose':True
}

mf = NEW_MF(R_temp,hyper_params)
test_set = mf.set_test(ratings_test)
result = mf.test()

Iteration : 10 ; Train RMSE = 0.9666 ; Test RMSE = 0.9807
Iteration : 20 ; Train RMSE = 0.9412 ; Test RMSE = 0.9623
Iteration : 30 ; Train RMSE = 0.9298 ; Test RMSE = 0.9552
Iteration : 40 ; Train RMSE = 0.9229 ; Test RMSE = 0.9515
Iteration : 50 ; Train RMSE = 0.9180 ; Test RMSE = 0.9493
Iteration : 60 ; Train RMSE = 0.9141 ; Test RMSE = 0.9478
Iteration : 70 ; Train RMSE = 0.9103 ; Test RMSE = 0.9467
Iteration : 80 ; Train RMSE = 0.9062 ; Test RMSE = 0.9457
Iteration : 90 ; Train RMSE = 0.9013 ; Test RMSE = 0.9446
Iteration : 100 ; Train RMSE = 0.8949 ; Test RMSE = 0.9430


# 4.5. MF 최적의 파라미터 찾기

최적의 상수값 K, iteration, alpha 값을 찾는 것이 필요.

1. **대략적인 최적의 K의 위치 찾기**
- 예를 들어, 50~260 사이에서 10의 간격으로 RMSE를 모두 계산해보면서 확인.
2. **K 주변 탐색으로 최적의 K 찾기** 
- 예를 들어, 60이 대략적인 최적의 K라면, 50~70 사이에서 1의 간격으로 모두 계산해보면서 확인.
- Overfitting을 방지하기 위해서는 테스트셋에 대한 RMSE도 확인하면서 탐색해야 함.
3. **주어진 K를 통해 최적의 iteration 선택**
- iteration 역시 너무 많아지면 Overfitting 발생. 따라서 테스트셋에 대한 RMSE도 같이 확인 필요.

결국 모델 내부에서 다양한 셔플 값들을 활용하고 있기 때문에, 한 번의 실험으로 결정할 수 있는 상수들은 아님. 

다중 실험을 반복하면서, 최적의 K, iteration 등을 결정하는 것이 합리적.

In [ ]:
# 최적의 K 값 찾기
results = []
index = []

R_temp = ratings.pivot(index='user_id',
                       columns='movie_id',
                       values='rating').fillna(0)
for K in range(50,261,10):
  print(f'K : {K}')
  hyper_params = {
      'K': K,
      'alpha' : 0.001,
      'beta' : 0.02,
      'iterations' : 300,
      'verbose' : True
  }
  mf = NEW_MF(R_temp,
              hyper_params)
  test_set = mf.set_test(ratings_test)
  result = mf.test()
  index.append(K)
  results.append(result)

In [ ]:
summary = []
for i in range(len(result)):
  RMSE = []
  for result in results[i]:
    RMSE.append(result[2])
  min = np.min(RMSE)
  j = RMSE.index(min)
  summary.append([index[i],j+1,RMSE[j]])